In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install hf
!hf auth login
!hf download microsoft/wavlm-base-plus-sv --local-dir /content/MyDrive/Data_Science_Project/Models/wavlm-base-plus-sv

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 752.88it/s]
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         ✓ Downloaded
  path: /content/MyDrive/Data_Science_Project/Models/wavlm-base-plus-sv
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            


In [7]:
DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

In [ ]:
!rsync -ah --progress {DATA_ROOT}/sinhala_audio.tar.gz /content/
!rsync -ah --progress {DATA_ROOT}/utt_spk_text.tsv /content/
!rsync -ah --progress {DATA_ROOT}/librispeech_clean100.tar.gz /content/
!rsync -ah --progress {DATA_ROOT}/rirs_noises.zip /content/
!rsync -ah --progress {DATA_ROOT}/musan.tar.gz /content/
!rsync -ah --progress {DATA_ROOT}/enchanced_zips/enhanced_audio_batch_20000_finish_2.tar.gz /content/

sending incremental file list
sinhala_audio.tar.gz
          4.83G 100%   48.29MB/s    0:01:35 (xfr#1, to-chk=0/1)
sending incremental file list
utt_spk_text.tsv
         16.15M 100%   68.63MB/s    0:00:00 (xfr#1, to-chk=0/1)
sending incremental file list
librispeech_clean100.tar.gz
          6.39G 100%   36.98MB/s    0:02:44 (xfr#1, to-chk=0/1)
sending incremental file list
rirs_noises.zip
          1.31G 100%   39.31MB/s    0:00:31 (xfr#1, to-chk=0/1)
sending incremental file list
musan.tar.gz
         11.09G 100%   45.52MB/s    0:03:52 (xfr#1, to-chk=0/1)


In [ ]:
!tar -xzf /content/sinhala_audio.tar.gz -C /content/
!rm /content/sinhala_audio.tar.gz

!tar -xzf /content/librispeech_clean100.tar.gz -C /content/
!rm /content/librispeech_clean100.tar.gz

!unzip -q /content/rirs_noises.zip -d /content/
!rm /content/rirs_noises.zip

!tar -xzf /content/musan.tar.gz -C /content/
!rm /content/musan.tar.gz

!tar -xzf /content/enhanced_audio_batch_20000_finish_2.tar.gz -C /content/
!rm /content/enhanced_audio_batch_20000_finish_2.tar.gz

In [ ]:
!pip install speechbrain torchaudio

In [ ]:
"""
PHASE 3b (v2) -- Degraded vs. Enhanced comparison.

KEY CHANGE FROM v1:
  v1 built trials from the full speaker/file universe, then tried to load
  enhanced audio for those files -- if the enhanced file was missing on disk
  (e.g. Phase 3a was interrupted for that condition), it was silently
  skipped, shrinking the enhanced branch's trial count while the degraded
  branch (regenerated on the fly, never missing) kept full coverage. That
  produced degenerate EER/FAR/FRR combinations for thin conditions.

  v2 inverts the flow, PER CONDITION:
    1. List the enhanced .wav files that actually exist on disk for this
       (language, condition).
    2. Map those files back to their original clean source file (by stem)
       to find which speaker they belong to.
    3. Build positive/negative trial pairs ONLY from files with an enhanced
       counterpart on disk.
    4. Degraded audio is regenerated from the ORIGINAL clean file using the
       same deterministic seed as Phase 3a, so both branches evaluate the
       exact same file set for that condition.

  This means: if a condition is thin, BOTH branches are equally thin (and
  it gets flagged via n_pos/n_neg + MIN_TRIALS), rather than one branch
  being thin and the other full-strength.
"""

import argparse
import glob
import itertools
import random
import zlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio

SAMPLE_RATE = 16000
MIN_TRIALS = 30  # below this, a row is flagged INSUFFICIENT_TRIALS rather than trusted

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier

CONDITIONS = [
    "clean",
    "reverb",
    "rirs_noise_0db",
    "musan_noise_0db",
    "musan_music_10db",
    "musan_speech_15db",
    "musan_speech_5db",
    "musan_speech_0db",
    "reverb+musan_noise_5db",
    "reverb+musan_speech_5db",
]

BABBLE_CONDITIONS = {"musan_speech_15db", "musan_speech_5db", "musan_speech_0db",
                      "reverb+musan_speech_5db"}

In [ ]:
# =====================================================================
# DATA DISCOVERY
# =====================================================================
def discover_sinhala_speakers(audio_dir, tsv_path):
    metadata = pd.read_csv(tsv_path, sep="\t", header=None,
                            names=["FileID", "UserID", "Transcription"])
    found_files = glob.glob(str(Path(audio_dir) / "**" / "*"), recursive=True)
    found_files = [f for f in found_files if f.lower().endswith((".wav", ".flac"))]
    stem_to_path = {Path(f).stem: f for f in found_files}
    metadata["path"] = metadata["FileID"].map(stem_to_path)
    available = metadata.dropna(subset=["path"])
    return available.groupby("UserID")["path"].apply(list).to_dict()


def discover_librispeech_speakers(root_dir):
    speaker_to_files = {}
    for speaker_dir in sorted(Path(root_dir).iterdir()):
        if speaker_dir.is_dir():
            files = glob.glob(str(speaker_dir / "**" / "*.flac"), recursive=True)
            if files:
                speaker_to_files[speaker_dir.name] = files
    return speaker_to_files


def discover_rir_files(rirs_root, max_pool=2000):
    files = glob.glob(str(Path(rirs_root) / "simulated_rirs" / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No RIR files found under {rirs_root}/simulated_rirs")
    if max_pool and len(files) > max_pool:
        files = random.Random(42).sample(files, max_pool)
    return files


def discover_generic_noise_files(root, subset):
    files = glob.glob(str(Path(root) / subset / "**" / "*.wav"), recursive=True)
    if not files:
        raise RuntimeError(f"No files found under {root}/{subset}")
    return files

In [ ]:
# =====================================================================
# ENHANCED-BACKED TRIAL BUILDING  (the core change)
# =====================================================================
def build_enhanced_backed_trials(speaker_to_files, enhanced_condition_dir,
                                  utts_per_speaker, seed):
    """
    Only include files that (a) exist as enhanced .wav files on disk for this
    condition AND (b) map back to a known original clean source file.
    Returns (positive_pairs, negative_pairs, eval_files, per_speaker_counts)
    where eval_files are ORIGINAL clean file paths (degraded audio is
    regenerated from these; enhanced audio is loaded from enhanced_condition_dir
    using the same stem).
    """
    stem_to_speaker_and_path = {}
    for spk, files in speaker_to_files.items():
        for f in files:
            stem_to_speaker_and_path[Path(f).stem] = (spk, f)

    enhanced_dir = Path(enhanced_condition_dir)
    if not enhanced_dir.exists():
        return [], [], [], {}

    enhanced_stems = {p.stem for p in enhanced_dir.glob("*.wav")}

    filtered_speaker_to_files = defaultdict(list)
    for stem in enhanced_stems:
        hit = stem_to_speaker_and_path.get(stem)
        if hit is not None:
            spk, original_path = hit
            filtered_speaker_to_files[spk].append(original_path)

    rng = random.Random(seed)
    eligible = {s: f for s, f in filtered_speaker_to_files.items() if len(f) >= 2}
    capped = {s: sorted(f)[:utts_per_speaker] for s, f in eligible.items()}

    per_speaker_counts = {s: len(f) for s, f in capped.items()}

    positive_pairs = []
    for spk, files in capped.items():
        positive_pairs.extend(itertools.combinations(files, 2))
    rng.shuffle(positive_pairs)

    speakers = list(capped.keys())
    file_to_speaker = {f: s for s, files in capped.items() for f in files}
    negative_pairs = []
    for f1, _ in positive_pairs:
        spk1 = file_to_speaker[f1]
        candidates = [s for s in speakers if s != spk1]
        if not candidates:
            continue
        other_spk = rng.choice(candidates)
        other_file = rng.choice(capped[other_spk])
        negative_pairs.append((f1, other_file))

    eval_files = list({f for p in positive_pairs for f in p} |
                       {f for p in negative_pairs for f in p})
    return positive_pairs, negative_pairs, eval_files, per_speaker_counts

In [ ]:
# =====================================================================
# AUDIO + DEGRADATION -- MUST MATCH phase3a_enhance_audio_fast.py EXACTLY
# =====================================================================
def load_mono_16k(path: str) -> torch.Tensor:
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    return waveform.squeeze(0)


def _rms(x):
    return torch.sqrt(torch.mean(x ** 2) + 1e-12)


def add_reverb(clean, rir_path):
    rir = load_mono_16k(rir_path)
    rir = rir / (torch.max(torch.abs(rir)) + 1e-8)
    augmented = torchaudio.functional.fftconvolve(clean, rir, mode="full")[: clean.shape[-1]]
    augmented = augmented.squeeze().to(dtype=clean.dtype)
    return augmented * (_rms(clean) / (_rms(augmented) + 1e-8))


def add_noise_at_snr(clean, noise_path, snr_db):
    noise = load_mono_16k(noise_path)
    if noise.shape[-1] < clean.shape[-1]:
        reps = int(np.ceil(clean.shape[-1] / noise.shape[-1]))
        noise = noise.repeat(reps)
    noise = noise[: clean.shape[-1]]
    target_noise_rms = _rms(clean) / (10 ** (snr_db / 20))
    noise = noise * (target_noise_rms / (_rms(noise) + 1e-8))
    return clean + noise


def apply_condition(clean, condition, rir_files, rirs_noise_files,
                     musan_noise_files, musan_music_files, musan_speech_files, rng):
    if condition == "clean":
        return clean
    if condition == "reverb":
        return add_reverb(clean, rng.choice(rir_files))
    if condition.startswith("rirs_noise_"):
        return add_noise_at_snr(clean, rng.choice(rirs_noise_files),
                                 float(condition.split("_")[-1].replace("db", "")))
    if condition.startswith("musan_noise_"):
        return add_noise_at_snr(clean, rng.choice(musan_noise_files),
                                 float(condition.split("_")[-1].replace("db", "")))
    if condition.startswith("musan_music_"):
        return add_noise_at_snr(clean, rng.choice(musan_music_files),
                                 float(condition.split("_")[-1].replace("db", "")))
    if condition.startswith("musan_speech_"):
        return add_noise_at_snr(clean, rng.choice(musan_speech_files),
                                 float(condition.split("_")[-1].replace("db", "")))
    if condition == "reverb+musan_noise_5db":
        return add_noise_at_snr(add_reverb(clean, rng.choice(rir_files)),
                                 rng.choice(musan_noise_files), 5.0)
    if condition == "reverb+musan_speech_5db":
        return add_noise_at_snr(add_reverb(clean, rng.choice(rir_files)),
                                 rng.choice(musan_speech_files), 5.0)
    raise ValueError(f"Unknown condition: {condition}")


def deterministic_condition_seed(condition: str, base_seed: int) -> int:
    return zlib.crc32(condition.encode("utf-8")) ^ base_seed


def per_file_rng(condition: str, file_path: str, base_seed: int) -> random.Random:
    cond_offset = deterministic_condition_seed(condition, base_seed)
    return random.Random(zlib.crc32(f"{condition}:{file_path}".encode()) ^ cond_offset)


# =====================================================================
# EMBEDDING + METRICS
# =====================================================================
def get_embedding(signal_tensor, model, device):
    signal = signal_tensor.unsqueeze(0).float().to(device)
    with torch.no_grad():
        emb = model.encode_batch(signal).squeeze(1)
        return F.normalize(emb, p=2, dim=-1)


def calculate_eer(pos_scores, neg_scores):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0, 0.0
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    thresholds = np.sort(np.concatenate([pos, neg]))
    min_diff, best_eer, best_thresh = float("inf"), 1.0, 0.0
    for t in thresholds:
        far, frr = np.mean(neg >= t), np.mean(pos < t)
        diff = abs(far - frr)
        if diff < min_diff:
            min_diff, best_eer, best_thresh = diff, (far + frr) / 2.0, t
    return best_eer * 100, best_thresh


def far_frr_at_threshold(pos_scores, neg_scores, threshold):
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    return (float(np.mean(neg >= threshold)) * 100,
            float(np.mean(pos < threshold)) * 100)


In [ ]:
# =====================================================================
# MAIN
# =====================================================================
def run_branch(branch_name, condition, positive_pairs, negative_pairs,
                eval_files, get_audio_fn, ecapa_model, device, clean_threshold_store,
                store_key):
    cache = {}
    n_missing = 0
    for key in eval_files:
        try:
            audio = get_audio_fn(key)
            if audio is None:
                n_missing += 1
                continue
            cache[key] = get_embedding(audio, ecapa_model, device)
        except Exception as e:
            n_missing += 1
            print(f"      [Warning] {branch_name}/{condition}/{key}: {e}")

    if n_missing > 0:
        print(f"      [{branch_name}/{condition}] {n_missing}/{len(eval_files)} files failed to load")

    cos = torch.nn.CosineSimilarity(dim=-1)
    pos_scores = [cos(cache[a], cache[b]).item()
                  for a, b in positive_pairs if a in cache and b in cache]
    neg_scores = [cos(cache[a], cache[b]).item()
                  for a, b in negative_pairs if a in cache and b in cache]

    eer, thresh = calculate_eer(pos_scores, neg_scores)
    if condition == "clean":
        clean_threshold_store[store_key] = thresh
    far, frr = far_frr_at_threshold(pos_scores, neg_scores,
                                     clean_threshold_store.get(store_key, thresh))
    return {"eer_retuned": eer, "far_at_clean_threshold": far,
            "frr_at_clean_threshold": frr, "n_pos": len(pos_scores), "n_neg": len(neg_scores)}


def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device.type.upper()}")

    print("\n=== Loading model ===")
    ecapa = EncoderClassifier.from_hparams(
        source=args["ecapa_model_path"],
        savedir="tmpdir_ecapa", run_opts={"device": str(device)},
    )

    print("\n=== Discovering original speech data ===")
    speakers_by_language = {}
    if args.get("sinhala_audio_dir") and args.get("sinhala_tsv"):
        speakers_by_language["Sinhala"] = discover_sinhala_speakers(
            args["sinhala_audio_dir"], args["sinhala_tsv"])
    if args.get("english_dir"):
        speakers_by_language["English"] = discover_librispeech_speakers(args["english_dir"])

    print("\n=== Discovering noise/RIR data ===")
    rir_files = discover_rir_files(args["rirs_root"], max_pool=args["max_rir_pool"])
    rirs_noise_files = discover_generic_noise_files(args["rirs_root"], "pointsource_noises")
    musan_noise_files = discover_generic_noise_files(args["musan_root"], "noise")
    musan_music_files = discover_generic_noise_files(args["musan_root"], "music")
    musan_speech_files = discover_generic_noise_files(args["musan_root"], "speech")

    clean_threshold_store = {}
    all_rows = []

    for language_name, speaker_to_files in speakers_by_language.items():
        print(f"\n{'='*20} {language_name} {'='*20}")
        clean_cache = {}

        for condition in CONDITIONS:
            enh_dir = Path(args["enhanced_dir"]) / language_name / condition
            if not enh_dir.exists() or not any(enh_dir.glob("*.wav")):
                print(f"--- [{language_name}] {condition} -> SKIPPED (no enhanced data on disk) ---")
                continue

            pos_pairs, neg_pairs, eval_files, per_speaker_counts = build_enhanced_backed_trials(
                speaker_to_files, enh_dir, args["utts_per_speaker"], args["seed"])

            n_speakers = len(per_speaker_counts)
            print(f"\n--- [{language_name}] {condition} "
                  f"(speakers_with_enhanced_data={n_speakers}, "
                  f"pos_pairs={len(pos_pairs)}, neg_pairs={len(neg_pairs)}) ---")

            if n_speakers < 2 or not pos_pairs or not neg_pairs:
                print(f"   [SKIP] Not enough enhanced coverage to build trials "
                      f"(need >=2 speakers with >=2 enhanced utts each).")
                all_rows.append({
                    "language": language_name, "condition": condition,
                    "degraded_eer": None, "degraded_far": None, "degraded_frr": None,
                    "degraded_n_pos": 0, "degraded_n_neg": 0,
                    "enhanced_eer": None, "enhanced_far": None, "enhanced_frr": None,
                    "enhanced_n_pos": 0, "enhanced_n_neg": 0,
                    "delta_eer": None, "verdict": "INSUFFICIENT_TRIALS",
                    "is_babble": condition in BABBLE_CONDITIONS,
                })
                continue

            def get_degraded(key, _cond=condition):
                if key not in clean_cache:
                    clean_cache[key] = load_mono_16k(key)
                rng = per_file_rng(_cond, key, args["seed"])
                return apply_condition(clean_cache[key], _cond, rir_files, rirs_noise_files,
                                        musan_noise_files, musan_music_files,
                                        musan_speech_files, rng)

            def get_enhanced(key, _cond=condition):
                enh_path = Path(args["enhanced_dir"]) / language_name / _cond / f"{Path(key).stem}.wav"
                if not enh_path.exists():
                    return None  # shouldn't happen: eval_files were built from this exact listing
                return load_mono_16k(str(enh_path))

            degraded_result = run_branch("degraded", condition, pos_pairs, neg_pairs, eval_files,
                                          get_degraded, ecapa, device, clean_threshold_store,
                                          ("degraded", language_name))
            enhanced_result = run_branch("enhanced", condition, pos_pairs, neg_pairs, eval_files,
                                          get_enhanced, ecapa, device, clean_threshold_store,
                                          ("enhanced", language_name))

            reliable = (degraded_result["n_pos"] >= MIN_TRIALS and degraded_result["n_neg"] >= MIN_TRIALS
                        and enhanced_result["n_pos"] >= MIN_TRIALS and enhanced_result["n_neg"] >= MIN_TRIALS)

            delta_eer = enhanced_result["eer_retuned"] - degraded_result["eer_retuned"]
            verdict = "HELPS" if delta_eer < -0.5 else ("HURTS" if delta_eer > 0.5 else "~same")
            if not reliable:
                verdict = "INSUFFICIENT_TRIALS"
            flag = "  <-- BABBLE" if condition in BABBLE_CONDITIONS else ""

            print(f"   Degraded EER={degraded_result['eer_retuned']:.2f}% "
                  f"(n_pos={degraded_result['n_pos']}, n_neg={degraded_result['n_neg']})  "
                  f"Enhanced EER={enhanced_result['eer_retuned']:.2f}% "
                  f"(n_pos={enhanced_result['n_pos']}, n_neg={enhanced_result['n_neg']})  "
                  f"Delta={delta_eer:+.2f}  [{verdict}]{flag}")

            all_rows.append({
                "language": language_name, "condition": condition,
                "degraded_eer": degraded_result["eer_retuned"],
                "degraded_far": degraded_result["far_at_clean_threshold"],
                "degraded_frr": degraded_result["frr_at_clean_threshold"],
                "degraded_n_pos": degraded_result["n_pos"], "degraded_n_neg": degraded_result["n_neg"],
                "enhanced_eer": enhanced_result["eer_retuned"],
                "enhanced_far": enhanced_result["far_at_clean_threshold"],
                "enhanced_frr": enhanced_result["frr_at_clean_threshold"],
                "enhanced_n_pos": enhanced_result["n_pos"], "enhanced_n_neg": enhanced_result["n_neg"],
                "delta_eer": delta_eer, "verdict": verdict,
                "is_babble": condition in BABBLE_CONDITIONS,
            })

    df = pd.DataFrame(all_rows)
    print("\n" + "=" * 130)
    print("PHASE 3b (v2) -- DEGRADED vs ENHANCED (DeepFilterNet3), ECAPA-TDNN, enhanced-backed trials")
    print("=" * 130)
    print(df.to_string(index=False))
    print("=" * 130)

    reliable_df = df[df["verdict"] != "INSUFFICIENT_TRIALS"]
    if not reliable_df.empty:
        babble_avg_delta = reliable_df[reliable_df["is_babble"]]["delta_eer"].mean()
        nonbabble_avg_delta = reliable_df[~reliable_df["is_babble"]
                                           & (reliable_df["condition"] != "clean")]["delta_eer"].mean()
        print(f"\nAverage EER delta (enhanced - degraded), babble conditions: {babble_avg_delta:+.2f}")
        print(f"Average EER delta (enhanced - degraded), non-babble conditions: {nonbabble_avg_delta:+.2f}")

    n_flagged = (df["verdict"] == "INSUFFICIENT_TRIALS").sum()
    if n_flagged > 0:
        print(f"\n[WARNING] {n_flagged} row(s) flagged INSUFFICIENT_TRIALS -- "
              f"these conditions need more completed Phase 3a enhancement before they're trustworthy:")
        print(df[df["verdict"] == "INSUFFICIENT_TRIALS"][["language", "condition"]].to_string(index=False))

    df.to_csv("phase3b_degraded_vs_enhanced_results_v2.csv", index=False)
    print("\nSaved: phase3b_degraded_vs_enhanced_results_v2.csv")


if __name__ == "__main__":
    args = {
        "sinhala_audio_dir": None,   # set to "/content/sinhala_audio" to include Sinhala
        "sinhala_tsv": None,         # set to "/content/utt_spk_text.tsv" to include Sinhala
        "english_dir": "/content/LibriSpeech/train-clean-100",
        "rirs_root": "/content/RIRS_NOISES",
        "musan_root": "/content/musan",
        "enhanced_dir": "/content/enhanced_audio",
        "ecapa_model_path": "/content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN",
        "utts_per_speaker": 4,
        "seed": 42,
        "max_rir_pool": 2000,
    }
    main(args)


In [ ]:
!python /content/phase3b_eval_enhanced_v2.py \
    --sinhala_audio_dir /content/sinhala_audio \
    --sinhala_tsv /content/utt_spk_text.tsv \
    --english_dir /content/LibriSpeech/train-clean-100 \
    --rirs_root /content/RIRS_NOISES \
    --musan_root /content/musan \
    --enhanced_dir /content/enhanced_audio \
    --ecapa_model_path /content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN

Device: CUDA

=== Loading model ===

embedding_model.ckpt: downloading bytes:  87% 72.3M/83.3M [00:01<00:00, 97.1MB/s, 5.88MB/s  ]
embedding_model.ckpt: downloading bytes: 100% 80.5M/80.5M [00:01<00:00, 49.5MB/s, 7.64MB/s  ]
embedding_model.ckpt: reconstructing file: 100% 83.3M/83.3M [00:01<00:00, 51.3MB/s, 8.04MB/s  ]
mean_var_norm_emb.ckpt: 100% 1.92k/1.92k [00:00<00:00, 11.0MB/s]

classifier.ckpt: downloading bytes:  77% 4.28M/5.53M [00:00<00:00, 7.43MB/s]
classifier.ckpt: downloading bytes: 100% 5.24M/5.24M [00:00<00:00, 8.98MB/s,  519kB/s  ]
classifier.ckpt: reconstructing file: 100% 5.53M/5.53M [00:00<00:00, 9.48MB/s,  548kB/s  ]
label_encoder.txt: 100% 129k/129k [00:00<00:00, 45.5MB/s]
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.

=== Discovering original speech data ===

=== Discovering noise/RIR data ===

==================== Sinhala ====================

--- [Sinhala] clean (speakers_with_enhanced_data=4

## Phase 3b (v2) — Degraded vs Enhanced (DeepFilterNet3), ECAPA-TDNN, Enhanced-Backed Trials

| Language | Condition | Degraded EER | Degraded FAR | Degraded FRR | Enhanced EER | Enhanced FAR | Enhanced FRR | Delta EER | Verdict | Is Babble |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Sinhala | clean | 4.46% | 4.46% | 4.46% | 7.67% | 7.67% | 7.67% | +3.21% | HURTS | False |
| Sinhala | reverb | 6.52% | 4.74% | 9.07% | 11.72% | 7.85% | 16.04% | +5.20% | HURTS | False |
| Sinhala | rirs_noise_0db | 8.12% | 3.77% | 17.09% | 11.82% | 10.74% | 12.97% | +3.70% | HURTS | False |
| Sinhala | musan_noise_0db | 9.14% | 3.77% | 18.48% | 12.59% | 10.88% | 14.16% | +3.45% | HURTS | False |
| Sinhala | musan_music_10db | 6.35% | 3.73% | 9.03% | 9.17% | 8.54% | 9.66% | +2.82% | HURTS | False |
| Sinhala | musan_speech_15db | 16.91% | 2.96% | 31.80% | 13.91% | 7.04% | 20.99% | -3.00% | HELPS | True |
| Sinhala | musan_speech_5db | 34.73% | 1.12% | 78.91% | 30.37% | 5.20% | 59.66% | -4.36% | HELPS | True |
| Sinhala | musan_speech_0db | 40.41% | 0.73% | 92.02% | 40.79% | 3.49% | 82.57% | +0.38% | ~same | True |
| Sinhala | reverb+musan_noise_5db | 10.08% | 3.56% | 23.50% | 14.05% | 8.82% | 20.78% | +3.97% | HURTS | False |
| Sinhala | reverb+musan_speech_5db | 38.77% | 1.01% | 91.00% | 39.23% | 3.66% | 83.96% | +0.45% | ~same | True |
| English | clean | 0.40% | 0.40% | 0.40% | 0.40% | 0.40% | 0.40% | 0.00% | ~same | False |
| English | reverb | 0.60% | 0.27% | 0.73% | 1.33% | 1.00% | 1.99% | +0.73% | HURTS | False |
| English | rirs_noise_0db | 1.66% | 0.33% | 4.32% | 4.78% | 4.52% | 4.91% | +3.12% | HURTS | False |
| English | musan_noise_0db | 1.26% | 0.13% | 3.78% | 4.12% | 3.32% | 4.52% | +2.86% | HURTS | False |
| English | musan_music_10db | 0.46% | 0.33% | 0.60% | 0.66% | 0.73% | 0.60% | +0.20% | ~same | False |
| English | musan_speech_15db | 1.13% | 0.27% | 1.86% | 1.00% | 0.40% | 1.26% | -0.13% | ~same | True |
| English | musan_speech_5db | 5.44% | 0.20% | 11.62% | 4.71% | 0.80% | 9.36% | -0.73% | HELPS | True |
| English | musan_speech_0db | 15.34% | 0.27% | 41.70% | 14.21% | 0.93% | 29.81% | -1.13% | HELPS | True |
| English | reverb+musan_noise_5db | 1.73% | 0.33% | 4.98% | 4.52% | 3.05% | 6.24% | +2.79% | HURTS | False |
| English | reverb+musan_speech_5db | 16.47% | 0.66% | 41.63% | 21.31% | 0.73% | 43.63% | +4.85% | HURTS | True |

In [24]:
!hf download speechbrain/spkrec-resnet-voxceleb --local-dir /content/speechbrain/spkrec-resnet-voxceleb

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/62.0M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/69.4M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/69.4M [00:00<?, ?B/s]

Fetching 11 files:   9% 1/11 [00:00<00:07,  1.34it/s]
Reconstructing (incomplete total...):   0% 1.74k/69.4M [00:00<8:29:20, 2.27kB/s]
Reconstructing (incomplete total...):   0% 1.79k/69.4M [00:00<8:29:21, 2.27kB/s]
Reconstructing (incomplete total...):   0% 1.79k/69.4M [00:00<8:29:23, 2.27kB/s]
Reconstructing (incomplete total...):   0% 6.62k/69.4M [00:00<8:29:21, 2.27kB/s]
Reconstructing (incomplete total...):   0% 6.62k/69.4M [00:00<8:29:22, 2.27kB/s]
Reconstructing (incomplete total...):   0% 8.25k/69.5M [00:00<8:30:07, 2.27kB/s]
Reconstructing (incomplete total...):   0% 113k/69.5M [00:00<8:29:39, 2.27kB/s] 
Reconstructing (incomplete total.

In [23]:
!rm -r /content/nvidia/

In [ ]:
!python /content/phase3b_eval_enhanced_v2.py \
    --sinhala_audio_dir /content/sinhala_audio \
    --sinhala_tsv /content/utt_spk_text.tsv \
    --english_dir /content/LibriSpeech/train-clean-100 \
    --rirs_root /content/RIRS_NOISES \
    --musan_root /content/musan \
    --enhanced_dir /content/enhanced_audio \
    --ecapa_model_path /content/speechbrain/spkrec-resnet-voxceleb

Device: CUDA

=== Loading model ===
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.

=== Discovering original speech data ===

=== Discovering noise/RIR data ===

==================== Sinhala ====================

--- [Sinhala] clean (speakers_with_enhanced_data=478, pos_pairs=2868, neg_pairs=2868) ---
   Degraded EER=4.50% (n_pos=2868, n_neg=2868)  Enhanced EER=7.71% (n_pos=2868, n_neg=2868)  Delta=+3.21  [HURTS]

--- [Sinhala] reverb (speakers_with_enhanced_data=478, pos_pairs=2868, neg_pairs=2868) ---
   Degraded EER=6.59% (n_pos=2868, n_neg=2868)  Enhanced EER=11.54% (n_pos=2868, n_neg=2868)  Delta=+4.95  [HURTS]

--- [Sinhala] rirs_noise_0db (speakers_with_enhanced_data=478, pos_pairs=2868, neg_pairs=2868) ---
   Degraded EER=8.44% (n_pos=2868, n_neg=2868)  Enhanced EER=11.68% (n_pos=2868, n_neg=2868)  Delta=+3.24  [HURTS]

--- [Sinhala] musan_noise_0db (speakers_with_enhanced_data=478, pos_pairs=2868, neg_pairs